In [0]:
%pip install kaggle

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.media_pipeline;
CREATE VOLUME IF NOT EXISTS workspace.media_pipeline.config;
CREATE VOLUME IF NOT EXISTS workspace.media_pipeline.bronze;

In [0]:
#Testing the Kaggle connection 
import os

with open("/Volumes/workspace/media_pipeline/config/access_token.txt") as f:
    os.environ["KAGGLE_API_TOKEN"] = f.read().strip()

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

files = api.dataset_list_files("loveall/clicks-conversion-tracking").files
print([f.name for f in files])

In [0]:
#Extraction cell
import csv, json, shutil, tempfile, logging
from datetime import date, datetime
from pathlib import Path

SOURCES = {
    "social_platform": "loveall/clicks-conversion-tracking",
    "daily_campaign_platform": "amirmotefaker/ab-testing-dataset",
    "multichannel_platform": "manishabhatt22/marketing-campaign-performance-dataset",
}
BRONZE = Path("/Volumes/workspace/media_pipeline/bronze")
ingest_date = date.today().isoformat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("extract")

def count_rows(p):
    with open(p, newline="", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in csv.reader(f)) - 1, 0)

results = []
for platform, slug in SOURCES.items():
    try:
        with tempfile.TemporaryDirectory() as tmp:
            api.dataset_download_files(slug, path=tmp, unzip=True, quiet=True)
            target = BRONZE / platform / f"ingest_date={ingest_date}"
            target.mkdir(parents=True, exist_ok=True)
            files = []
            for src in Path(tmp).rglob("*.csv"):
                shutil.copy(src, target / src.name)
                files.append({"file": src.name, "rows": count_rows(src)})
        if not files:
            raise RuntimeError("no CSV files in download")
        log.info("%s: %s", platform, files)
        results.append({"platform": platform, "slug": slug, "status": "success", "files": files})
    except Exception as e:
        log.error("%s failed: %s", platform, e)
        results.append({"platform": platform, "slug": slug, "status": "failed", "error": str(e)})

manifest_dir = BRONZE / "_manifests"
manifest_dir.mkdir(exist_ok=True)
(manifest_dir / f"manifest_{ingest_date}.json").write_text(json.dumps(
    {"ingest_date": ingest_date, "run_at": datetime.now().isoformat(timespec="seconds"), "sources": results}, indent=2))
print(json.dumps(results, indent=2))

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/media_pipeline/bronze"))

In [0]:
#Checking the manifest 
import json
from datetime import date

manifest = json.loads(open(
    f"/Volumes/workspace/media_pipeline/bronze/_manifests/manifest_{date.today().isoformat()}.json").read())
for s in manifest["sources"]:
    print(s["platform"], s["status"], s.get("files", s.get("error")))

In [0]:
#Profiling each source 
from pyspark.sql import functions as F

BRONZE = "/Volumes/workspace/media_pipeline/bronze"
paths = {
    "social": f"{BRONZE}/social_platform/*/KAG_conversion_data.csv",
    "daily_control": f"{BRONZE}/daily_campaign_platform/*/control_group.csv",
    "multichannel": f"{BRONZE}/multichannel_platform/*/*.csv",
}

for name, path in paths.items():
    df = spark.read.option("header", True).csv(path)
    print(f"\n===== {name}: {df.count()} rows =====")
    df.printSchema()
    display(df.limit(5))
    display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

In [0]:
BRONZE = "/Volumes/workspace/media_pipeline/bronze"

# Daily: read with the correct delimiter, keep the file name to tell control from test
daily = (spark.read.option("header", True).option("sep", ";")
         .csv(f"{BRONZE}/daily_campaign_platform/*/*.csv")
         .withColumn("source_file", F.col("_metadata.file_name")))
display(daily)
display(daily.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in daily.columns]))

# Social: conversions without clicks, and approved conversions above total
social = spark.read.option("header", True).csv(f"{BRONZE}/social_platform/*/*.csv")
print("Conversions with zero clicks:",
      social.filter((F.col("Clicks").cast("int") == 0) & (F.col("Total_Conversion").cast("int") > 0)).count())
print("Approved > Total:",
      social.filter(F.col("Approved_Conversion").cast("int") > F.col("Total_Conversion").cast("int")).count())

# Multichannel: what do the text-number columns really look like?
multi = spark.read.option("header", True).csv(f"{BRONZE}/multichannel_platform/*/*.csv")
display(multi.select("Acquisition_Cost", "Duration", "Conversion_Rate", "ROI").limit(10))

In [0]:
import re
from pyspark.sql import functions as F

BRONZE = "/Volumes/workspace/media_pipeline/bronze"
SOURCES = {
    "bronze_social":       (f"{BRONZE}/social_platform/*/*.csv", ","),
    "bronze_daily":        (f"{BRONZE}/daily_campaign_platform/*/*.csv", ";"),
    "bronze_multichannel": (f"{BRONZE}/multichannel_platform/*/*.csv", ","),
}

def safe_name(col):
    return re.sub(r"[^0-9a-zA-Z]+", "_", col).strip("_")

for table, (path, sep) in SOURCES.items():
    df = (spark.read.option("header", True).option("sep", sep).csv(path)
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))
    df = df.toDF(*[c if c.startswith("_") else safe_name(c) for c in df.columns])
    df.write.mode("overwrite").saveAsTable(f"workspace.media_pipeline.{table}")
    print(f"{table}: {spark.table(f'workspace.media_pipeline.{table}').count()} rows")

In [0]:
%sql

DROP TABLE IF EXISTS workspace.media_pipeline.silver_daily;

CREATE TABLE workspace.media_pipeline.silver_daily
(
    platform            STRING          NOT NULL,
    campaign_name       STRING,
    report_date         DATE,
    spend               DECIMAL(12,2),
    impressions         INT,
    reach               INT,
    clicks              INT,
    searches            INT,
    view_content        INT,
    add_to_cart         INT,
    purchases           INT,
    is_incomplete       BOOLEAN         NOT NULL,
    has_funnel_anomaly  BOOLEAN         NOT NULL,
    _source_file        STRING
);

In [0]:
%sql

INSERT INTO workspace.media_pipeline.silver_daily
(
    platform,
    campaign_name,
    report_date,
    spend,
    impressions,
    reach,
    clicks,
    searches,
    view_content,
    add_to_cart,
    purchases,
    is_incomplete,
    has_funnel_anomaly,
    _source_file
)
SELECT
    'daily_campaign_platform' AS platform,

    Campaign_Name AS campaign_name,

    to_date(`Date`, 'd.MM.yyyy') AS report_date,

    try_cast(Spend_USD AS DECIMAL(12,2)) AS spend,

    try_cast(of_Impressions AS INT) AS impressions,

    try_cast(Reach AS INT) AS reach,

    try_cast(of_Website_Clicks AS INT) AS clicks,

    try_cast(of_Searches AS INT) AS searches,

    try_cast(of_View_Content AS INT) AS view_content,

    try_cast(of_Add_to_Cart AS INT) AS add_to_cart,

    try_cast(of_Purchase AS INT) AS purchases,

    CASE
        WHEN try_cast(of_Impressions AS INT) IS NULL
        THEN TRUE
        ELSE FALSE
    END AS is_incomplete,

    CASE
        WHEN try_cast(of_Purchase AS INT) >
             try_cast(of_Add_to_Cart AS INT)
          OR try_cast(of_Add_to_Cart AS INT) >
             try_cast(of_View_Content AS INT)
        THEN TRUE
        ELSE FALSE
    END AS has_funnel_anomaly,

    _source_file

FROM workspace.media_pipeline.bronze_daily;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.media_pipeline.silver_daily
(
    platform            STRING          NOT NULL,
    campaign_name       STRING,
    report_date         DATE,
    spend               DECIMAL(12,2),
    impressions         INT,
    reach               INT,
    clicks              INT,
    searches            INT,
    view_content        INT,
    add_to_cart         INT,
    purchases           INT,
    is_incomplete       BOOLEAN         NOT NULL,
    has_funnel_anomaly  BOOLEAN         NOT NULL,
    _source_file        STRING
);

In [0]:
%sql
SELECT
  COUNT(*)                                AS total_rows,
  COUNT_IF(report_date IS NULL)           AS bad_dates,
  COUNT_IF(is_incomplete)                 AS incomplete_rows,
  COUNT_IF(has_funnel_anomaly)            AS funnel_anomalies,
  MIN(report_date)                        AS first_day,
  MAX(report_date)                        AS last_day
FROM workspace.media_pipeline.silver_daily;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.media_pipeline.bronze_daily;

In [0]:
%sql DESCRIBE HISTORY workspace.media_pipeline.silver_daily

In [0]:
%sql
SELECT COUNT(*) FROM workspace.media_pipeline.bronze_daily;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_daily AS
SELECT
  'daily_campaign_platform'                 AS platform,
  Campaign_Name                             AS campaign_name,
  to_date(`Date`, 'd.MM.yyyy')              AS report_date,
  CAST(Spend_USD AS DECIMAL(12,2))          AS spend,
  CAST(of_Impressions AS INT)               AS impressions,
  CAST(Reach AS INT)                        AS reach,
  CAST(of_Website_Clicks AS INT)            AS clicks,
  CAST(of_Searches AS INT)                  AS searches,
  CAST(of_View_Content AS INT)              AS view_content,
  CAST(of_Add_to_Cart AS INT)               AS add_to_cart,
  CAST(of_Purchase AS INT)                  AS purchases,
  of_Impressions IS NULL                    AS is_incomplete,
  COALESCE(CAST(of_Purchase AS INT) > CAST(of_Add_to_Cart AS INT)
        OR CAST(of_Add_to_Cart AS INT) > CAST(of_View_Content AS INT), FALSE)
                                            AS has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.bronze_daily;

In [0]:
%sql DESCRIBE HISTORY workspace.media_pipeline.silver_daily

In [0]:
%sql
describe workspace.media_pipeline.bronze_daily

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_social AS
SELECT
  'social_platform'                 AS platform,
  ad_id                             AS ad_id,
  xyz_campaign_id                           AS campaign_id,
  fb_campaign_id                            AS platform_campaign_id,
  age                                       AS age_group,
  gender                                    AS gender,
  CAST(interest AS INT)                      AS interest_id,
  CAST(impressions AS INT)                  AS impressions,
  CAST(Clicks AS INT)                                       AS clicks,
  CAST(spent AS DECIMAL(12,2))              AS spend,
  CAST(Total_Conversion AS INT)          AS conversions,
  CAST(Approved_Conversion AS INT)               AS approved_conversions,
  CAST(Clicks AS INT) = 0 AND CAST(Total_Conversion AS INT)>0 AS is_view_through,
  _source_file
FROM workspace.media_pipeline.bronze_social;

In [0]:
%sql 
SELECT
  COUNT(*)                     AS total_rows,
  COUNT_IF(is_view_through)    AS view_through_rows,
  MIN(platform)                AS platform_value,
  typeof(MIN(clicks))          AS clicks_type
FROM workspace.media_pipeline.silver_social;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_multichannel AS
SELECT
  'multichannel_platform'                 AS platform,
  campaign_id                           AS campaign_id,
  Company                                 AS company,
  Campaign_Type                           AS campaign_type,
  Target_Audience                         AS target_audience,
  Channel_Used                            AS channel,
  CAST(REPLACE(Duration, ' days', '') AS INT) AS duration_days,
  to_date(`Date`, 'yyyy-MM-dd') AS report_date,
  CAST(REPLACE(REPLACE(Acquisition_Cost, '$', ''), ',', '') AS DECIMAL(12,2)) AS spend,
  CAST(impressions AS INT)                  AS impressions,
  CAST(Clicks AS INT)                       AS clicks,
  CAST(Conversion_Rate AS DECIMAL(5,4))              AS conversion_rate,
  CAST(ROI AS DECIMAL(6,2))              AS roi,
  CAST(Engagement_Score AS INT)                      AS engagement_score,
  Location                                 AS location,
  Language                                 AS language,
  Customer_Segment                         AS customer_segment,
  _source_file
FROM workspace.media_pipeline.bronze_multichannel;

In [0]:
%sql
SELECT
  COUNT(*)                           AS total_rows,
  COUNT_IF(spend IS NULL)            AS null_spend,
  COUNT_IF(duration_days IS NULL)    AS null_duration,
  COUNT_IF(report_date IS NULL)      AS null_dates,
  MIN(report_date)                   AS first_day,
  MAX(report_date)                   AS last_day,
  MIN(spend)                         AS min_spend,
  MAX(spend)                         AS max_spend
FROM workspace.media_pipeline.silver_multichannel;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_campaign_performance AS
 SELECT
 platform,
 campaign_name as campaign_id,
   report_date,
  spend,
  impressions,
  clicks,
  purchases AS conversions,
  is_incomplete,
 FALSE AS is_view_through,
 has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.silver_daily
UNION ALL 
SELECT
 platform,
 campaign_id,
 NULL as report_date,
 spend,
 impressions,
 clicks,
 conversions,
 FALSE AS is_incomplete,
 is_view_through,
 FALSE as has_funnel_anomaly,
 _source_file
FROM workspace.media_pipeline.silver_social
UNION ALL 

SELECT
  platform,
 campaign_id,
 report_date,
 spend,
 impressions,
 clicks,
 Null AS conversions,
  FALSE AS is_incomplete,
 FALSE AS is_view_through,
 FALSE as has_funnel_anomaly,
 _source_file
FROM workspace.media_pipeline.silver_multichannel;

In [0]:
%sql
SELECT platform, COUNT(*) AS rows, SUM(spend) AS total_spend
FROM workspace.media_pipeline.silver_campaign_performance
GROUP BY platform
ORDER BY platform;

In [0]:
%sql
SELECT 'daily' AS src, SUM(spend) FROM workspace.media_pipeline.silver_daily
UNION ALL
SELECT 'social', SUM(spend) FROM workspace.media_pipeline.silver_social
UNION ALL
SELECT 'multichannel', SUM(spend) FROM workspace.media_pipeline.silver_multichannel;

In [0]:
%sql
DESCRIBE TABLE workspace.media_pipeline.silver_campaign_performance;